This code uses synthetic CMS data published by Synthea to generate logistic models assessing the probability of having received a flu vaccine based on the number of times that a patient had a medical 'encounter', how many other vaccines they received, their age group, and associated county. It also creates a secondary ML logit model to allow the user to predict the probability of having received the flu vaccine for any given combination of encounters, other vaccines, and age. 

### Code Setup - Import Needed Libraries and Set Local Variables if Needed

In [ ]:
#Python 3.14.5

import numpy as np #2.4.6
import pandas as pd #3.0.3
import statsmodels.formula.api as smf #0.14.6
from sklearn.linear_model import LogisticRegression #1.8.0

path = r"INSERT PATH\synthea_sample_data_csv_nov2021" #Insert path to dataset here
#Data Source: https://synthea.mitre.org/downloads

________________

### DATA IMPORT

In [2]:
#Count number of encounters per patient per year
encounters = pd.read_csv(fr"{path}\csv\encounters.csv")
encounters['YEAR'] = pd.to_datetime(encounters['START']).dt.year
encounters_counts = (encounters.groupby(['PATIENT', 'YEAR']).size().reset_index(name='ENCOUNTER_COUNT'))

#Count number of flu vaccines and other vaccines per patient per year
immunizations = pd.read_csv(fr"{path}\csv\immunizations.csv")
immunizations['YEAR'] = pd.to_datetime(immunizations['DATE']).dt.year
immunizations['FLU_VACCINE'] = np.where(immunizations['DESCRIPTION'] == 'Influenza  seasonal  injectable  preservative free', 1, 0)
flu_immunizations = immunizations[immunizations['FLU_VACCINE'] == 1]
other_immunizations = immunizations[immunizations['FLU_VACCINE'] == 0]
flu_counts = (flu_immunizations.groupby(['PATIENT', 'YEAR']).size().reset_index(name='FLU_VACCINE_COUNT'))
other_counts = (other_immunizations.groupby(['PATIENT', 'YEAR']).size().reset_index(name='OTHER_VACCINE_COUNT'))

#Merge datasets together
merged_data = encounters_counts.merge(other_counts, left_on=['PATIENT', 'YEAR'], right_on=['PATIENT', 'YEAR'], how='outer')
merged_data = merged_data.merge(flu_counts, left_on=['PATIENT', 'YEAR'], right_on=['PATIENT', 'YEAR'], how='outer')
merged_data['OTHER_VACCINE_COUNT'] = merged_data['OTHER_VACCINE_COUNT'].fillna(0)
merged_data['FLU_VACCINE_INDICATOR'] = np.where(merged_data['FLU_VACCINE_COUNT'] > 0, 1, 0)

#Read in patient data for age and county information, merge with main dataset, and create age groups and clean county data
patients = pd.read_csv(fr"{path}\csv\patients.csv")
merged_data = merged_data.merge(patients[['Id', 'BIRTHDATE', 'COUNTY']], left_on='PATIENT', right_on='Id', how='left')
merged_data.drop(columns=['Id'], inplace=True)
merged_data['BIRTHDATE'] = pd.to_datetime(merged_data['BIRTHDATE'])
merged_data['AGE'] = ((pd.to_datetime(merged_data['YEAR'].astype(str) + '-01-01') - merged_data['BIRTHDATE']).dt.days // 365) + 1
merged_data['AGE_GROUP'] = pd.cut(merged_data['AGE'], bins=[-1, 18, 35, 50, 65, 120], labels=['0-18', '19-35', '36-50', '51-65', '66+'])
merged_data['COUNTY'] = merged_data['COUNTY'].fillna('Unknown')
merged_data

,PATIENT,YEAR,ENCOUNTER_COUNT,OTHER_VACCINE_COUNT,FLU_VACCINE_COUNT,FLU_VACCINE_INDICATOR,BIRTHDATE,COUNTY,AGE,AGE_GROUP
0,00126cb9-8460-4747-e302-c3609684531e,2005,2,0.0,NaN,0,1987-05-30,Bristol County,18,0-18
1,00126cb9-8460-4747-e302-c3609684531e,2006,1,0.0,NaN,0,1987-05-30,Bristol County,19,19-35
2,00126cb9-8460-4747-e302-c3609684531e,2007,1,0.0,NaN,0,1987-05-30,Bristol County,20,19-35
3,00126cb9-8460-4747-e302-c3609684531e,2008,1,0.0,NaN,0,1987-05-30,Bristol County,21,19-35
4,00126cb9-8460-4747-e302-c3609684531e,2009,1,0.0,NaN,0,1987-05-30,Bristol County,22,19-35
...,...,...,...,...,...,...,...,...,...,...
25981,ff9f14e4-d241-71fe-a501-2199e39aa79a,2017,1,0.0,1.0,1,2003-07-26,Middlesex County,14,0-18
25982,ff9f14e4-d241-71fe-a501-2199e39aa79a,2018,2,0.0,1.0,1,2003-07-26,Middlesex County,15,0-18
25983,ff9f14e4-d241-71fe-a501-2199e39aa79a,2019,1,1.0,1.0,1,2003-07-26,Middlesex County,16,0-18
25984,ff9f14e4-d241-71fe-a501-2199e39aa79a,2020,1,0.0,1.0,1,2003-07-26,Middlesex County,17,0-18


Run logit model to see how total number of encounters, other vaccines received, age group, and county affected likelihood of receiving flu vaccine

In [3]:
#Run logistic regression model to predict flu vaccination based on number of encounters, number of other vaccines, age group, and county - conditional on being observed in the dataset (i.e. having at least one encounter or vaccine)
flu_vax_model = smf.logit(formula='FLU_VACCINE_INDICATOR ~ ENCOUNTER_COUNT + OTHER_VACCINE_COUNT + AGE_GROUP + COUNTY', data=merged_data).fit()
flu_vax_model.summary()

Optimization terminated successfully.
         Current function value: 0.526250
         Iterations 7


<class 'statsmodels.iolib.summary.Summary'>
"""
                             Logit Regression Results                            
=================================================================================
Dep. Variable:     FLU_VACCINE_INDICATOR   No. Observations:                25986
Model:                             Logit   Df Residuals:                    25967
Method:                              MLE   Df Model:                           18
Date:                   Thu, 21 May 2026   Pseudo R-squ.:                  0.1914
Time:                           12:38:29   Log-Likelihood:                -13675.
converged:                          True   LL-Null:                       -16911.
Covariance Type:               nonrobust   LLR p-value:                     0.000
==============================================================================================
                                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------
Intercept                     -0.7754      0.068    -11.367      0.000      -0.909      -0.642
AGE_GROUP[T.19-35]            -1.4321      0.045    -31.725      0.000      -1.521      -1.344
AGE_GROUP[T.36-50]            -0.8611      0.045    -19.186      0.000      -0.949      -0.773
AGE_GROUP[T.51-65]            -0.1602      0.046     -3.507      0.000      -0.250      -0.071
AGE_GROUP[T.66+]               0.0210      0.053      0.396      0.692      -0.083       0.125
COUNTY[T.Berkshire County]    -0.4304      0.122     -3.523      0.000      -0.670      -0.191
COUNTY[T.Bristol County]      -0.0896      0.078     -1.146      0.252      -0.243       0.064
COUNTY[T.Essex County]         0.0895      0.075      1.187      0.235      -0.058       0.237
COUNTY[T.Franklin County]     -0.6773      0.108     -6.268      0.000      -0.889      -0.465
COUNTY[T.Hampden County]      -0.2453      0.085     -2.889      0.004      -0.412      -0.079
COUNTY[T.Hampshire County]     0.3613      0.112      3.218      0.001       0.141       0.581
COUNTY[T.Middlesex County]     0.0895      0.069      1.292      0.196      -0.046       0.225
COUNTY[T.Nantucket County]     2.3074      1.071      2.154      0.031       0.208       4.407
COUNTY[T.Norfolk County]      -0.1423      0.075     -1.907      0.056      -0.288       0.004
COUNTY[T.Plymouth County]     -0.1337      0.079     -1.692      0.091      -0.289       0.021
COUNTY[T.Suffolk County]       0.1763      0.076      2.318      0.020       0.027       0.325
COUNTY[T.Worcester County]    -0.0417      0.077     -0.542      0.588      -0.192       0.109
ENCOUNTER_COUNT                0.2888      0.008     35.607      0.000       0.273       0.305
OTHER_VACCINE_COUNT            0.8593      0.029     29.279      0.000       0.802       0.917
==============================================================================================
"""

Create ML Logit and function to predict likelihood of having received flu vaccine for any requested combination of encounter count, other vaccine count, and age

In [4]:
# Define and fit a machine learning logit
Xvars=merged_data[['ENCOUNTER_COUNT','OTHER_VACCINE_COUNT','AGE']]
flu_vax_logit2=LogisticRegression(penalty='l2', random_state=0)
flu_vax_model2=flu_vax_logit2.fit(Xvars,merged_data['FLU_VACCINE_INDICATOR'])

def flu_vax_bot2(encounter_count=4, other_vaccine_count=3, age=10):
    # Put the inputs into the exact dataframe the logit expects to see
    input_df=pd.DataFrame([[encounter_count, other_vaccine_count, age]],
    columns=['ENCOUNTER_COUNT','OTHER_VACCINE_COUNT','AGE'])
    features=['ENCOUNTER_COUNT', 'OTHER_VACCINE_COUNT', 'AGE']

    # Calculate the predicted probability
    flux_vax_prediction=flu_vax_model2.predict_proba(input_df[features])[0,1]

    # Put the probability into a nice format and return the recommendation
    prob_string=str(np.round(flux_vax_prediction,3)*100)[0:4]
    if flux_vax_prediction > 0.5:
        return "[Encounters = " + str(encounter_count) + " | Other Vaccines = " + str(other_vaccine_count) + " | Age = " + str(age) + "] Person most likely to receive flu vaccine (Pr="+prob_string+"%)"
    if flux_vax_prediction <=0.5:
        return "[Encounters = " + str(encounter_count) + " | Other Vaccines = " + str(other_vaccine_count) + " | Age = " + str(age) + "] Person most likely NOT to receive flu vaccine (Pr="+prob_string+"%)"

c:\Users\mattv\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Predict likelihood of receiving flu vaccine based on total encounters, total other vaccinations, and age

In [5]:
print(flu_vax_bot2(encounter_count=1, other_vaccine_count=1, age=10))
print(flu_vax_bot2(encounter_count=1, other_vaccine_count=1, age=53))
print(flu_vax_bot2(encounter_count=2, other_vaccine_count=0, age=80))
print(flu_vax_bot2(encounter_count=5, other_vaccine_count=4, age=25))

[Encounters = 1 | Other Vaccines = 1 | Age = 10] Person most likely NOT to receive flu vaccine (Pr=40.2%)
[Encounters = 1 | Other Vaccines = 1 | Age = 53] Person most likely NOT to receive flu vaccine (Pr=48.1%)
[Encounters = 2 | Other Vaccines = 0 | Age = 80] Person most likely NOT to receive flu vaccine (Pr=37.0%)
[Encounters = 5 | Other Vaccines = 4 | Age = 25] Person most likely to receive flu vaccine (Pr=97.6%)
